# 🔢 [Colab 실습] 양자화 첫걸음 — 순수 파이썬으로 밑바닥부터

**온디바이스 AI 프로그래밍 · PyTorch 양자화 미니랩(PTQ 7단계) 들어가기 전 준비 실습**

| 항목 | 내용 |
| --- | --- |
| 대상 | 양자화를 처음 접하는 분 (딥러닝 프레임워크 지식 불필요) |
| 도구 | **순수 파이썬 + numpy + matplotlib만** 사용 — PyTorch 없음! |
| 환경 | Google Colab CPU 런타임 (GPU 불필요) |
| 진행 | 위에서부터 셀을 하나씩 실행 (`Shift + Enter`) — 앞 셀의 함수를 뒤에서 재사용합니다 |

## 이 실습의 목표

PyTorch의 `quantize`, `Observer`, `prepare`, `convert` 같은 함수들은 사실 **아주 단순한 산수**를 감싼 포장지입니다.
이 실습에서는 그 포장지를 벗기고, 양자화의 모든 부품을 **여러분 손으로 직접** 만듭니다.
마지막에는 프레임워크 없이 만든 미니 신경망을 통째로 INT8로 바꿔 봅니다.

## 로드맵

| Part | 주제 | 만드는 것 |
| --- | --- | --- |
| 1 | 숫자 하나의 무게 — FP32는 왜 4바이트인가 | 바이트 직접 열어보기 |
| 2 | 눈금자의 발명 — scale | `quantize` 한 줄 손계산 |
| 3 | quantize 함수 완성 — clip까지 | 나만의 양자화 함수 |
| 4 | 정수만으로 곱하기 — INT8 연산의 핵심 정리 | INT8 내적·행렬곱 |
| 5 | 미니 뉴런 양자화 — 캘리브레이션의 탄생 | 손으로 만든 Observer |
| 6 | 🏁 종합: 미니 분류기 통째로 INT8 만들기 | 프레임워크 없는 PTQ |
| 7 | 정리 — PyTorch PTQ 7단계와의 대응표 | 다음 실습으로 가는 지도 |

> 💡 각 Step의 **✅ 확인**을 스스로 점검하고, **✏️ 직접 해보기**는 코드를 고쳐 실험해 보세요.
> 수식이 나오면 겁먹지 마세요 — 전부 곱셈·나눗셈·반올림뿐입니다.


---
# Part 0. 환경 준비

설치할 것은 없습니다. Colab 기본 패키지만 씁니다. (한글 그래프 폰트만 잠깐 설정)

In [ ]:
import struct
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
rng = np.random.default_rng(42)

# 그래프 한글 폰트 (실패해도 실습 진행에는 지장 없음)
try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"],
                   capture_output=True, timeout=120)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rc("font", family="NanumGothic")
    print("한글 폰트 설정 완료")
except Exception as e:
    print("한글 폰트 생략(그래프 한글이 □로 보일 수 있음):", e)
plt.rc("axes", unicode_minus=False)

print("numpy", np.__version__, "| 준비 완료 🚀")

---
# Part 1. 숫자 하나의 무게 — FP32는 왜 4바이트인가

"모델을 INT8로 양자화하면 크기가 1/4"이라는 말을 자주 듣습니다.
그 1/4이 어디서 오는지, **숫자 한 개를 바이트 단위로 열어서** 직접 확인합시다.

### Step 1-1. FP32 숫자 하나를 바이트로 열어보기

파이썬의 `struct` 모듈은 숫자가 메모리에 실제로 어떻게 저장되는지 보여줍니다.

In [ ]:
x = 0.7234                      # 신경망 가중치 하나라고 상상하세요

fp32_bytes = struct.pack('f', x)     # 'f' = float32 형식으로 포장
print(f"값 {x} 를 FP32로 저장하면:")
print(f"  바이트 수 : {len(fp32_bytes)}바이트")
print(f"  실제 내용 : {fp32_bytes.hex(' ')}  (16진수)")
print()
print("→ 실수 하나 = 4바이트(32비트). 이 4바이트 안에 부호·지수·가수가 들어 있습니다.")

### Step 1-2. INT8은 1바이트 — 대신 256개 값만 표현

INT8은 8비트 = 1바이트입니다. $2^8 = 256$가지, 부호를 포함하면 **−128 ~ +127**만 표현할 수 있습니다.

In [ ]:
q = 92                          # 나중에 0.7234가 변신하게 될 정수 (미리보기!)

int8_bytes = struct.pack('b', q)     # 'b' = signed 8bit
print(f"정수 {q} 를 INT8로 저장하면:")
print(f"  바이트 수 : {len(int8_bytes)}바이트")
print(f"  실제 내용 : {int8_bytes.hex()}")
print()
print(f"INT8 표현 범위: {-2**7} ~ {2**7-1}  (총 {2**8}가지)")
print(f"FP32 대비 크기: 1/4  ← '모델 크기 1/4'의 정체가 바로 이것!")

# 범위 밖 값을 넣으면 어떻게 될까?
try:
    struct.pack('b', 200)
except struct.error as e:
    print(f"\n200을 INT8에 넣으면? → 에러: {e}")
    print("→ 127을 넘는 값은 담을 수조차 없다 — 이 한계가 Part 3의 'clip'으로 이어집니다.")

### Step 1-3. 모델 규모로 환산 — 1/4의 위력 체감

가중치가 1,100만 개인 ResNet-18 규모로 계산해 봅시다.

> ✏️ **직접 해보기:** `n_params`를 여러분이 아는 다른 모델 크기로 바꿔 보세요.
> (참고: BlackSwan NPU의 모델 크기 제한은 약 30MB입니다 — INT8이 아니면 들어가기 힘든 이유!)

In [ ]:
n_params = 11_000_000            # ResNet-18 급

fp32_mb = n_params * 4 / 1024 / 1024
int8_mb = n_params * 1 / 1024 / 1024
print(f"파라미터 {n_params:,}개 기준")
print(f"  FP32 모델: {fp32_mb:6.1f} MB")
print(f"  INT8 모델: {int8_mb:6.1f} MB   (4배 절감)")
print()
print(f"NPU 모델 제한이 30MB라면 → FP32는 {'탑재 불가 ❌' if fp32_mb>30 else '탑재 가능'} / INT8은 {'탑재 가능 ✅' if int8_mb<=30 else '탑재 불가'}")

> **✅ Part 1 확인**
> - [ ] FP32 한 개 = 4바이트, INT8 한 개 = 1바이트임을 바이트를 열어 확인했다
> - [ ] INT8은 −128~127, 총 256가지만 표현함을 안다
> - [ ] "모델 1/4"이 단순히 4바이트→1바이트에서 옴을 계산했다
>
> **다음 질문:** 그런데 0.7234 같은 실수를 어떻게 −128~127 정수 세계로 옮기지?
> → Part 2에서 **눈금자(scale)** 를 발명합니다.

---
# Part 2. 눈금자의 발명 — scale

### Step 2-1. 문제를 정확히 이해하기

우리가 풀 문제:

```text
실수 세계:  -1.0 ────────── 0 ────────── +1.0   (연속, 무한히 촘촘)
정수 세계:  -127 ────────── 0 ────────── +127   (이산, 딱 255칸)
```

**지도의 축척**과 똑같습니다. 서울(실제 거리 수십 km)을 A4 종이에 그리려면
"1cm = 1km" 같은 **축척 하나**를 정하면 되죠. 양자화의 축척이 바로 **scale**입니다.

$$scale = \frac{r_{max}}{q_{max}} = \frac{\text{실수 범위의 최대값}}{127}$$

### Step 2-2. 손계산: 숫자 하나 양자화하기

실수 범위가 ±1.0이라고 하면 (r_max = 1.0):

In [ ]:
r_max = 1.0
q_max = 127
scale = r_max / q_max
print(f"scale = {r_max} / {q_max} = {scale:.6f}")
print(f"→ 의미: 정수 눈금 1칸 = 실수 {scale:.6f} 만큼")
print()

# 양자화: 실수를 scale로 나눠 '몇 칸인지' 세고 반올림
x = 0.7234
q = round(x / scale)
print(f"x = {x}")
print(f"q = round({x} / {scale:.6f}) = round({x/scale:.2f}) = {q}")
print()
print(f"✅ 실수 {x} 는 정수 {q} 가 되었다 — Part 1에서 미리 본 92의 정체!")

### Step 2-3. 복원(dequantize)과 오차 — 반올림의 대가

정수를 다시 실수로 되돌리려면 곱하면 됩니다: $\hat{x} = q \times scale$

되돌린 값 $\hat{x}$은 원래 $x$와 **완전히 같지 않습니다** — round()에서 버린 소수점이 오차가 됩니다.

In [ ]:
x_hat = q * scale
err = x - x_hat
print(f"복원값 x̂ = {q} × {scale:.6f} = {x_hat:.6f}")
print(f"원본     x = {x}")
print(f"오차 x−x̂  = {err:+.6f}")
print()
print(f"오차의 이론적 최대 = scale/2 = {scale/2:.6f}  (반올림은 최대 반 칸까지만 틀림)")
assert abs(err) <= scale/2 + 1e-12
print("✅ 오차가 반 칸 이내임을 확인 — 양자화 오차의 정체는 '반올림'이다")

### Step 2-4. 여러 숫자로 왕복 실험 — 표로 감 잡기

> ✏️ **직접 해보기:** `samples`에 아무 숫자나 추가해 보세요. 0.0과 1.0(딱 끝값)도 넣어보세요.
> **1.3처럼 r_max를 넘는 값**도 넣어보세요 — q가 127을 넘어버립니다! (Part 3의 예고편)

In [ ]:
samples = [0.0, 0.1, 0.5, 0.7234, -0.9, 1.0, 1.3]   # ← 마지막 1.3은 범위 밖!

print(f"{'x':>8} | {'x/scale':>9} | {'q':>5} | {'복원 x̂':>9} | {'오차':>9} | 비고")
print("-" * 62)
for x in samples:
    q = round(x / scale)
    x_hat = q * scale
    note = "⚠️ q가 127 초과!" if abs(q) > 127 else ""
    print(f"{x:>8.4f} | {x/scale:>9.2f} | {q:>5} | {x_hat:>9.4f} | {x-x_hat:>+9.4f} | {note}")
print()
print("💡 1.3 → q=165: INT8에 담을 수 없는 값이 나왔다. Part 3에서 해결합니다.")

> **✅ Part 2 확인**
> - [ ] scale = r_max/127 을 지도의 축척에 비유해 설명할 수 있다
> - [ ] quantize = `round(x/scale)`, dequantize = `q×scale` 두 줄을 손으로 계산했다
> - [ ] 오차 ≤ scale/2 (반올림 반 칸)임을 확인했다
> - [ ] 범위 밖 값(1.3)에서 q가 127을 넘는 문제를 목격했다

---
# Part 3. quantize 함수 완성 — clip까지

### Step 3-1. 순수 파이썬으로 배열 양자화 (첫 버전)

리스트의 모든 값을 양자화하는 함수를 for 루프로 짭니다. 아직 clip은 없습니다.

In [ ]:
def quantize_v1(values, scale):
    """1차 시도: 반올림만 (clip 없음)"""
    return [round(v / scale) for v in values]

def dequantize(q_values, scale):
    return [q * scale for q in q_values]

weights = [0.12, -0.55, 0.98, -0.31, 0.7234, -1.0]
q_list = quantize_v1(weights, scale)
back = dequantize(q_list, scale)

print("원본  :", [f"{v:+.4f}" for v in weights])
print("INT8  :", q_list)
print("복원  :", [f"{v:+.4f}" for v in back])
mean_err = sum(abs(a-b) for a,b in zip(weights, back)) / len(weights)
print(f"평균 오차: {mean_err:.6f}  (scale/2 = {scale/2:.6f} 이내 ✅)")

### Step 3-2. 사고 실험: 범위 밖 값이 섞이면? → clip의 탄생

실제 데이터엔 예상 범위를 벗어나는 **이상치**가 섞입니다. v1 함수에 넣으면 INT8에 담을 수 없는
q가 나옵니다. 해결책은 단순합니다 — **경계에서 잘라내기(clip)**:

$$q = \mathrm{clip}\big(\mathrm{round}(x / scale),\ -128,\ +127\big)$$

In [ ]:
bad_weights = [0.5, 1.8, -2.3, 0.9]      # 1.8, -2.3은 r_max=1.0 밖!

q_bad = quantize_v1(bad_weights, scale)
print("v1 결과:", q_bad, " ← 229, -292는 INT8에 저장 불가 💥")

def quantize_v2(values, scale):
    """완성형: 반올림 + clip(-128, 127)"""
    out = []
    for v in values:
        q = round(v / scale)
        q = max(-128, min(127, q))       # ← clip 한 줄이 전부!
        out.append(q)
    return out

q_ok = quantize_v2(bad_weights, scale)
back = dequantize(q_ok, scale)
print("v2 결과:", q_ok, " ← 전부 INT8 범위 안 ✅")
print()
print(f"{'x':>7} | {'q':>5} | {'복원':>8} | {'오차':>8} | 해석")
print("-"*52)
for x, q, b in zip(bad_weights, q_ok, back):
    kind = "잘림(clip)! 오차 큼" if abs(round(x/scale)) > 127 else "반올림 오차만"
    print(f"{x:>7.2f} | {q:>5} | {b:>8.4f} | {x-b:>+8.4f} | {kind}")
print()
print("💡 clip된 값의 오차는 반올림 오차와 차원이 다르게 큽니다.")
print("   'clip을 얼마나 허용할 것인가'가 Part 5 캘리브레이션의 핵심 고민이 됩니다.")

### Step 3-3. numpy 버전 — 같은 논리, 배열 한 번에

앞으로 큰 배열을 다루므로 numpy로 옮깁니다. **결과가 순수 파이썬 버전과 완전히 같은지** 검증합니다.
r_max를 데이터에서 자동으로 구하는 기능도 넣습니다 (지금은 '데이터의 최대 절대값'을 쓰면 됩니다).

In [ ]:
def quantize(x, r_max=None):
    """완성형 numpy 양자화.
    x: 배열, r_max: 실수 범위(없으면 max|x| 자동) → (int8 배열, scale)"""
    x = np.asarray(x, dtype=np.float64)
    if r_max is None:
        r_max = np.abs(x).max()          # 데이터가 스스로 범위를 말해줌
    s = r_max / 127 if r_max > 0 else 1.0
    q = np.clip(np.round(x / s), -128, 127).astype(np.int8)
    return q, s

# 검증: 순수 파이썬 v2와 동일한가?
q_np, s_np = quantize(bad_weights, r_max=1.0)
assert list(q_np) == quantize_v2(bad_weights, s_np), "두 구현이 달라요!"
print("✅ numpy 구현 == 순수 파이썬 구현 (같은 산수라는 증거)")
print()

# 자동 r_max 사용 예
w = rng.normal(0, 0.4, 8)
qw, sw = quantize(w)                      # r_max = max|w| 자동
print("원본   :", np.round(w, 3))
print("INT8   :", qw)
print(f"scale  : {sw:.5f}  (자동 결정: max|w|={np.abs(w).max():.3f} ÷ 127)")
print("복원   :", np.round(qw * sw, 3))

### Step 3-4. 눈으로 보기 — 256칸 격자에 스냅되는 분포

1,000개의 값을 양자화하면 무슨 일이 생기는지 히스토그램으로 봅니다.
복원값은 **scale 간격의 격자 위에만** 존재하게 됩니다 — 연속이 이산이 되는 순간입니다.

> 📎 함께 보기: HTML 애니메이션 「FP32→INT8 양자화 매핑」— 이 격자가 슬라이더로 움직입니다.

In [ ]:
data = rng.normal(0, 0.35, 1000)
qd, sd = quantize(data)
restored = qd * sd

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
axes[0].hist(data, bins=120)
axes[0].set_title("원본 FP32 분포 (연속)")
axes[1].hist(restored, bins=120, color="#e8a33d")
axes[1].set_title(f"양자화→복원 분포 (scale={sd:.4f} 격자에만 존재)")
for ax in axes: ax.set_xlabel("값"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"서로 다른 복원값 개수: {len(np.unique(restored))}개  (최대 256개까지만 가능)")
print(f"평균 반올림 오차: {np.abs(data-restored).mean():.6f}  (이론 한계 scale/2={sd/2:.6f})")

> **✅ Part 3 확인**
> - [ ] clip이 왜 필요한지(INT8에 담을 수 없는 q) 설명할 수 있다
> - [ ] 완성형 `quantize()`를 순수 파이썬과 numpy로 각각 만들었고 결과가 같음을 검증했다
> - [ ] 복원값이 scale 격자 위에만 존재함을 히스토그램으로 확인했다
>
> **다음 질문:** 양자화한 정수로 **곱셈·덧셈**을 해도 될까? 신경망은 결국 곱하고 더하는 기계인데…
> → Part 4에서 INT8 연산의 핵심 정리를 증명합니다.

---
# Part 4. 정수만으로 곱하기 — INT8 연산의 핵심 정리

### Step 4-1. 종이에 먼저 증명하기

두 실수 $a = q_a \cdot s_a$, $b = q_b \cdot s_b$ 의 곱은:

$$a \times b = (q_a s_a)(q_b s_b) = \underbrace{(q_a \times q_b)}_{\text{정수 곱!}} \times \underbrace{(s_a s_b)}_{\text{상수}}$$

**곱셈의 무거운 부분은 정수끼리 하고, scale은 마지막에 한 번만 곱하면 된다** — 이것이 INT8 연산의 전부입니다.
덧셈도 마찬가지: scale이 같은 정수들은 그냥 더하면 됩니다. 그래서 내적(곱하고 더하기)이 통째로 정수 연산이 됩니다.

In [ ]:
a, b = 0.62, -0.38

qa, sa = quantize([a]); qb, sb = quantize([b])
qa, qb = int(qa[0]), int(qb[0])

true_prod = a * b
int_prod  = (qa * qb) * (sa * sb)         # 정수 곱 → 마지막에 scale 한 번

print(f"a={a} → q_a={qa} (s_a={sa:.5f})")
print(f"b={b} → q_b={qb} (s_b={sb:.5f})")
print()
print(f"진짜 곱     a×b        = {true_prod:.6f}")
print(f"정수 경로  q_a·q_b·s_a·s_b = {qa}×{qb}×{sa*sb:.8f} = {int_prod:.6f}")
print(f"오차: {abs(true_prod-int_prod):.6f}")
print()
print("✅ 실수 곱셈을 '정수 곱 + scale 한 번'으로 대체했다 — NPU가 INT8 MAC만으로 버티는 이유!")

### Step 4-2. 내적(dot product)을 INT8로 — 신경망 연산의 최소 단위

뉴런 하나의 계산 $y = \sum_i w_i x_i$ 를 정수 경로로 수행합니다.
**누산(acc)은 파이썬 int로** — 정수 곱들을 전부 더한 뒤, scale은 딱 한 번 곱합니다.

In [ ]:
w = rng.normal(0, 0.4, 16)
x = rng.normal(0, 0.8, 16)

qw, sw = quantize(w)
qx, sx = quantize(x)

# 정수 누산 (순수 파이썬 int — 넘칠 걱정 없음)
acc = 0
for i in range(16):
    acc += int(qw[i]) * int(qx[i])
y_int8 = acc * (sw * sx)                  # scale은 마지막에 한 번!

y_true = float(np.dot(w, x))
rel = abs(y_int8 - y_true) / (abs(y_true) + 1e-12) * 100
print(f"정수 누산 acc          = {acc}  (아직 scale 안 곱한 순정수)")
print(f"INT8 경로 y = acc·s_w·s_x = {y_int8:.6f}")
print(f"FP 정답   y            = {y_true:.6f}")
print(f"상대 오차: {rel:.2f}%")
print()
print("💡 곱 16번 + 덧셈 15번이 전부 정수 연산 — 하드웨어에선 이게 전력 20배 절감이 됩니다.")

### Step 4-3. 행렬곱 전체를 INT8 파이프라인으로

이제 x@W 행렬곱(가상 NPU 노트북과 같은 행벡터 규약)을 통째로 정수 경로로 처리하고 오차를 잽니다.

In [ ]:
X = rng.normal(0, 0.8, (4, 16))           # 입력 4개 (행벡터)
W = rng.normal(0, 0.4, (16, 8))           # 뉴런 8개

qX, sX = quantize(X)
qW, sW = quantize(W)

acc32 = qX.astype(np.int32) @ qW.astype(np.int32)   # ① 정수 행렬곱 (int32 누산)
Y_int8 = acc32 * (sX * sW)                          # ② scale 한 번
Y_true = X @ W

rel = np.abs(Y_int8 - Y_true).mean() / np.abs(Y_true).mean() * 100
print(f"acc32 dtype: {acc32.dtype} | 값 범위: {acc32.min()} ~ {acc32.max()}")
print(f"INT8 경로 vs FP 평균 상대 오차: {rel:.2f}%")
print(f"메모리: FP32 {X.nbytes+W.nbytes}B → INT8 {qX.nbytes+qW.nbytes}B (4배 절감)")
print()
print("✅ '오차 ~1% 주고 메모리 4배·전력 큰 폭 절감' — 양자화라는 거래의 실체입니다.")

### Step 4-4. 잠깐, 누산기는 왜 int32일까? (맛보기)

INT8 곱 하나의 최대는 $127 \times 127 = 16{,}129$. 하지만 K번 **누적**하면 훨씬 커집니다.

In [ ]:
K = 512
worst = 127 * 127 * K
print(f"최악의 경우 누적값: 127 × 127 × {K} = {worst:,}")
print(f"int16 최대(32,767)의 {worst/32767:.0f}배 → int16 누산기는 터진다!")
print(f"int32 최대({2**31-1:,})에는 여유 → 그래서 'INT8 곱 + INT32 누산'이 표준")
print()
print("📎 이 overflow를 직접 터뜨려 보는 실험은 「가상 NPU」 노트북 Part 6-3에 있습니다.")

> **✅ Part 4 확인**
> - [ ] $(q_a s_a)(q_b s_b) = q_a q_b \cdot s_a s_b$ 정리를 종이에 쓸 수 있다
> - [ ] 내적·행렬곱을 '정수 연산 + 마지막 scale 한 번'으로 수행했고 오차가 ~1%임을 확인했다
> - [ ] 누산기가 int32여야 하는 이유를 숫자로 말할 수 있다

---
# Part 5. 미니 뉴런 양자화 — 캘리브레이션의 탄생

### Step 5-1. 순수 파이썬 뉴런 만들기 (FP 기준선)

뉴런 하나: $y = \mathrm{ReLU}(w \cdot x + b)$

In [ ]:
w_n = [0.4, -0.7, 0.2]
b_n = 0.1

def neuron_fp(x):
    z = sum(wi * xi for wi, xi in zip(w_n, x)) + b_n
    return max(0.0, z)                    # ReLU

test_inputs = [[1.0, 0.5, -0.3], [0.2, -1.1, 0.8], [-0.5, 0.3, 1.2], [2.0, -0.2, 0.1]]
print("FP 뉴런 출력 (기준선):")
for x in test_inputs:
    print(f"  x={x} → y={neuron_fp(x):.4f}")

### Step 5-2. weight만 양자화해 보기 — 쉬운 절반

weight는 **모델 안에 이미 들어있는 고정된 숫자**입니다. 범위(max|w|)를 그 자리에서 알 수 있으니
양자화가 쉽습니다.

In [ ]:
qw_n, sw_n = quantize(w_n)
qb = round(b_n / sw_n)                    # bias도 같은 scale로 (간단화)

def neuron_wq(x):
    """weight만 INT8, 입력은 아직 FP"""
    z = sum(int(qwi) * sw_n * xi for qwi, xi in zip(qw_n, x)) + qb * sw_n
    return max(0.0, z)

print(f"w={w_n} → q={list(map(int,qw_n))} (scale={sw_n:.5f})")
print()
print(f"{'입력':>22} | {'FP':>8} | {'W양자화':>8} | 오차")
for x in test_inputs:
    a, b_ = neuron_fp(x), neuron_wq(x)
    print(f"{str(x):>22} | {a:>8.4f} | {b_:>8.4f} | {a-b_:+.5f}")
print()
print("✅ weight 양자화는 오차가 작다 — 범위를 정확히 알고 시작했기 때문.")

### Step 5-3. 문제 발견: activation의 범위는 미리 알 수 없다!

입력(그리고 중간 활성값)도 INT8로 바꾸려면 scale이 필요하고, scale에는 **r_max**가 필요합니다.
그런데 activation은 **어떤 입력이 들어오느냐에 따라 달라지는 값**입니다. 모델 파일 어디에도 안 적혀 있죠.

**해결책 = 캘리브레이션**: 대표 입력 몇 개를 미리 흘려보내며 활성값의 범위를 **관찰**해서 r_max를 정한다.
이 '관찰하는 코드'가 PyTorch에서 **Observer**라고 불리는 것의 정체입니다.

### Step 5-4. 손으로 만드는 Observer

In [ ]:
class MyObserver:
    """PyTorch Observer의 5줄짜리 본질: 지나가는 값의 min/max만 기록"""
    def __init__(self):
        self.min_seen = float("inf")
        self.max_seen = float("-inf")
    def watch(self, value):
        self.min_seen = min(self.min_seen, value)
        self.max_seen = max(self.max_seen, value)
    def r_max(self):
        return max(abs(self.min_seen), abs(self.max_seen))

# 캘리브레이션: 대표 입력들을 흘려보내며 '입력값'의 범위 관찰
calib_data = [rng.normal(0, 0.8, 3).tolist() for _ in range(50)]   # 대표 입력 50개

obs = MyObserver()
for x in calib_data:
    for xi in x:
        obs.watch(xi)                     # 값은 안 바꾸고 보기만!

r_max_act = obs.r_max()
s_act = r_max_act / 127
print(f"Observer 관찰 결과: min={obs.min_seen:.3f}, max={obs.max_seen:.3f}")
print(f"→ activation r_max = {r_max_act:.3f}, scale = {s_act:.5f}")
print()
print("💡 prepare() = 이런 Observer를 모델 곳곳에 심기 / calibrate = 데이터 흘리며 watch() 호출")
print("   convert() = 관찰 끝난 r_max로 scale 확정 — PyTorch 함수 3개의 정체가 전부 이것!")

### Step 5-5. 완전 INT8 뉴런 + 나쁜 캘리브레이션 실험

이제 weight와 입력을 모두 INT8로 처리하는 뉴런을 완성하고,
**캘리브레이션 데이터가 나쁘면**(실전과 다른 좁은 범위) 무슨 일이 나는지 실험합니다.

> 📎 함께 보기: HTML 애니메이션 「Observer 캘리브레이션」 — 이 실험의 시각화 버전입니다.

In [ ]:
def neuron_int8(x, s_in):
    """weight·입력 모두 INT8 경로"""
    qx = [max(-128, min(127, round(xi / s_in))) for xi in x]      # 입력 양자화(+clip)
    acc = sum(int(qwi) * qxi for qwi, qxi in zip(qw_n, qx))       # 정수 누산
    z = acc * (sw_n * s_in) + b_n                                 # scale은 한 번
    return max(0.0, z)

# 실전 입력 200개로 평가 (일부러 큰 값도 섞인 분포)
eval_inputs = [rng.normal(0, 0.8, 3).tolist() for _ in range(200)]

def mean_err(s_in):
    errs = [abs(neuron_fp(x) - neuron_int8(x, s_in)) for x in eval_inputs]
    return sum(errs) / len(errs)

# 좋은 캘리브레이션: 실전과 같은 분포 50개로 관찰 (위에서 이미 함)
err_good = mean_err(s_act)

# 나쁜 캘리브레이션: 좁은 분포(작은 값들만) 50개로 관찰
obs_bad = MyObserver()
for x in [rng.normal(0, 0.15, 3).tolist() for _ in range(50)]:
    for xi in x: obs_bad.watch(xi)
s_bad = obs_bad.r_max() / 127
err_bad = mean_err(s_bad)

# 참고: clip이 실제로 얼마나 발생하나
clip_rate = np.mean([abs(round(xi/s_bad)) > 127 for x in eval_inputs for xi in x]) * 100

print(f"{'캘리브레이션':>12} | {'r_max':>7} | {'평균 출력 오차':>12}")
print(f"{'좋음(대표성O)':>11} | {s_act*127:>7.3f} | {err_good:>12.5f}")
print(f"{'나쁨(좁은범위)':>11} | {s_bad*127:>7.3f} | {err_bad:>12.5f}   ← {err_bad/err_good:.0f}배 악화!")
print(f"\n나쁜 scale에서 실전 입력의 {clip_rate:.0f}%가 clip됨 — 오차 폭증의 범인")
print()
print("✅ '캘리브레이션 데이터는 실전을 대표해야 한다'는 규칙을 직접 증명했습니다.")

> **✅ Part 5 확인**
> - [ ] weight 양자화는 쉽고 activation 양자화에는 캘리브레이션이 필요한 이유를 구분해 말할 수 있다
> - [ ] Observer를 5줄로 직접 만들었다 (min/max 기록이 전부)
> - [ ] 나쁜 캘리브레이션 → clip 다발 → 오차 폭증의 인과를 실험으로 확인했다
>
> ✏️ **직접 해보기:** `obs_bad`의 캘리브레이션 분포를 0.15 → 0.4, 0.6으로 바꿔가며
> 오차가 어떻게 회복되는지 관찰하세요.

---
# Part 6. 🏁 종합 프로젝트 — 미니 분류기를 통째로 INT8로

프레임워크 없이, 지금까지 만든 부품만으로 **진짜 PTQ**를 수행합니다.

**과제 모델**: 4×4 픽셀 이미지가 숫자 **0인지 1인지** 구분하는 초소형 분류기 (16입력 → 2출력).

### Step 6-1. 데이터와 모델 준비 (이 셀은 실행만 하면 됩니다)

작은 numpy 학습 루프로 분류기를 준비합니다 — 오늘의 주제는 학습이 아니라 **양자화**이므로 내용은 몰라도 됩니다.

In [ ]:
# ── 4×4 '0'과 '1' 패턴 + 노이즈로 데이터 생성 ──
BASE0 = np.array([[1,1,1,1],[1,0,0,1],[1,0,0,1],[1,1,1,1]], float)   # ㅁ 모양
BASE1 = np.array([[0,0,1,0],[0,1,1,0],[0,0,1,0],[0,1,1,1]], float)   # 1 모양

def make_data(n):
    Xs, ys = [], []
    for _ in range(n):
        label = rng.integers(0, 2)
        img = (BASE1 if label else BASE0).copy()
        img += rng.normal(0, 0.25, (4,4))          # 밝기 노이즈
        Xs.append(img.reshape(-1)); ys.append(label)
    return np.array(Xs), np.array(ys)

Xtr, ytr = make_data(400)
Xte, yte = make_data(200)

# ── 초소형 로지스틱 분류기 학습 (16 → 2) ──
W_fc = rng.normal(0, 0.1, (16, 2)); b_fc = np.zeros(2)
for epoch in range(300):
    logits = Xtr @ W_fc + b_fc
    p = np.exp(logits - logits.max(1, keepdims=True))
    p /= p.sum(1, keepdims=True)
    onehot = np.eye(2)[ytr]
    gW = Xtr.T @ (p - onehot) / len(Xtr)
    gb = (p - onehot).mean(0)
    W_fc -= 0.5 * gW; b_fc -= 0.5 * gb

def acc_of(predict_fn):
    pred = np.array([predict_fn(x) for x in Xte])
    return (pred == yte).mean() * 100

def predict_fp(x):
    return int(np.argmax(x @ W_fc + b_fc))

acc_fp = acc_of(predict_fp)
print(f"학습 완료 — FP32 분류기 테스트 정확도: {acc_fp:.1f}%  (기준선)")

# 예시 이미지 보기
fig, axes = plt.subplots(1, 6, figsize=(9, 1.8))
for ax, (img, y) in zip(axes, zip(Xte[:6], yte[:6])):
    ax.imshow(img.reshape(4,4), cmap="gray"); ax.set_title(f"정답 {y}", fontsize=9); ax.axis("off")
plt.suptitle("테스트 이미지 예시 (4×4)"); plt.tight_layout(); plt.show()

### Step 6-2. PTQ 절차 그대로: 관찰 → scale 확정 → INT8 추론기 완성

PyTorch가 해주던 일을 우리 부품으로 수행합니다:
① weight 양자화(범위 즉시 확정) → ② Observer로 입력 범위 캘리브레이션 → ③ 완전 INT8 추론 함수

In [ ]:
# ① weight 양자화
qW_fc, sW_fc = quantize(W_fc)

# ② 캘리브레이션: '학습 데이터 일부'를 대표 데이터로 사용 (실전 규칙과 동일)
obs_in = MyObserver()
for x in Xtr[:100]:
    for v in x: obs_in.watch(float(v))
s_in = obs_in.r_max() / 127
print(f"② 캘리브레이션: 입력 r_max={obs_in.r_max():.3f} → s_in={s_in:.5f} (100개 관찰)")

# ③ 완전 INT8 추론기
def predict_int8(x):
    qx = np.clip(np.round(x / s_in), -128, 127).astype(np.int32)
    acc = qx @ qW_fc.astype(np.int32)                 # 정수 행렬·벡터 곱
    logits = acc * (s_in * sW_fc) + b_fc              # scale 한 번 + bias
    return int(np.argmax(logits))

acc_q = acc_of(predict_int8)
size_fp = W_fc.astype(np.float32).nbytes   # FP32 기준 (numpy 기본은 float64라서 명시)
size_q  = qW_fc.nbytes
print(f"\n════ PTQ 결과 리포트 ════")
print(f"정확도  : FP32 {acc_fp:.1f}%  →  INT8 {acc_q:.1f}%   (손실 {acc_fp-acc_q:+.1f}%p)")
print(f"모델크기: {size_fp}B → {size_q}B  (4배 절감)")
print()
verdict = "✅ 합격 — 이대로 배포!" if acc_fp - acc_q <= 2 else "⚠️ 손실 큼 — QAT 검토 (교안 결정 규칙)"
print(f"교안 결정 규칙(손실 ≤ 2%p면 PTQ 채택): {verdict}")

### Step 6-3. 리포트 과제 — 직접 실험하고 표를 채우세요

| 실험 | 조건 | INT8 정확도 | 관찰 |
| --- | --- | --- | --- |
| 기준 | 위 설정 그대로 | | |
| 실험1 | 캘리브레이션 100개 → **5개**만 | | |
| 실험2 | 캘리브레이션을 `rng.normal(0, 0.05, 16)` 같은 **엉뚱한 데이터**로 | | |
| 실험3 | 노이즈를 0.25 → 0.6으로 키운 테스트셋 | | |

**분석 질문 (2~3문장씩):**
1. 실험1에서 정확도가 유지됐다면 왜일까요? (힌트: 이 문제의 입력 분포는 얼마나 단순한가)
2. 실험2에서 무슨 일이 났나요? clip 관점에서 설명하세요.
3. "캘리브레이션 데이터 300~500장"이라는 실전 권장치는 어떤 상황을 대비한 걸까요?

In [ ]:
# ✏️ 실험 공간 — 예시 (실험2):
obs_weird = MyObserver()
for _ in range(100):
    for v in rng.normal(0, 0.05, 16): obs_weird.watch(float(v))
s_weird = obs_weird.r_max() / 127

def predict_weird(x):
    qx = np.clip(np.round(x / s_weird), -128, 127).astype(np.int32)
    return int(np.argmax(qx @ qW_fc.astype(np.int32) * (s_weird * sW_fc) + b_fc))

print(f"실험2 (엉뚱한 캘리브레이션): 정확도 {acc_of(predict_weird):.1f}%  (기준 {acc_q:.1f}%)")
clip_pct = np.mean([abs(round(v/s_weird))>127 for x in Xte[:50] for v in x])*100
print(f"→ 실전 입력의 {clip_pct:.0f}%가 clip됨 — 그런데 정확도가 버틸 수도 있습니다!?")

# 겉보기(정답 맞히기)는 버텨도 속(출력값)은 망가졌음을 확인:
def logits_of(x, s):
    qx = np.clip(np.round(x / s), -128, 127).astype(np.int32)
    return qx @ qW_fc.astype(np.int32) * (s * sW_fc) + b_fc
logit_err = np.mean([np.abs(logits_of(x, s_weird) - (x @ W_fc + b_fc)).mean() for x in Xte[:50]])
logit_err_good = np.mean([np.abs(logits_of(x, s_in) - (x @ W_fc + b_fc)).mean() for x in Xte[:50]])
print(f"출력(logit) 왜곡: 좋은 캘리브레이션 {logit_err_good:.4f} vs 엉뚱한 캘리브레이션 {logit_err:.4f} ({logit_err/logit_err_good:.0f}배)")
print()
print("💡 이 문제는 0/1 패턴 차이가 워낙 커서 argmax(정답 고르기)가 clip을 버텨냅니다.")
print("   하지만 출력값 자체는 크게 왜곡됨 — 클래스가 많거나 점수 차가 미세한 실전 모델이라면?")
print("   → 분석 질문 2·3에서 이어서 생각해 보세요. 실험1·3도 직접 작성해 보세요!")

---
# Part 7. 정리 — 오늘 만든 부품 ↔ PyTorch PTQ 7단계

다음 실습(PyTorch 양자화 미니랩)에서 만날 함수들은 전부 오늘 만든 부품의 산업용 버전입니다.

| 오늘 직접 만든 것 | PyTorch PTQ 7단계에서의 이름 |
| --- | --- |
| `scale = r_max / 127` 손계산 | qconfig가 정하는 양자화 방식의 핵심 상수 |
| `quantize()` (round + clip) | `torch.quantize_per_tensor` 내부 동작 |
| `MyObserver` (min/max 기록 5줄) | `MinMaxObserver` — ③ `prepare()`가 심는 그것 |
| 대표 입력 흘려보내기 | ④ Calibration 루프 (`for x in data: model(x)`) |
| r_max 확정 → INT8 추론기 완성 | ⑤ `convert()` |
| FP vs INT8 정확도 비교 + 2%p 규칙 | ⑥ 정확도 검증 → PTQ/QAT 결정 |
| 정수 곱 + 마지막 scale 한 번 | INT8 커널(fbgemm)이 하드웨어에서 하는 일 |

오늘 다루지 **않은** 것 (다음 실습에서 만납니다):
- **zero_point** (비대칭 양자화): 오늘은 0을 중심에 둔 대칭만 — ReLU 출력처럼 한쪽으로 쏠린 분포엔 비대칭이 유리
- **per-channel scale**: 오늘은 배열 전체가 scale 하나(per-tensor) — 채널별 맞춤은 「Per-Tensor vs Per-Channel」 애니메이션 참고
- **Fuse(Conv+BN+ReLU 병합)**, **QAT**: PyTorch 노트북 Part 2·6에서

## ✏️ 심화 도전 과제 (선택)

1. **zero_point 구현하기**: $q = \mathrm{round}(x/s) + z$ 형태의 비대칭 양자화 함수를 만들고,
   ReLU 출력(모두 양수) 데이터에서 대칭 방식과 오차를 비교하세요.
2. **INT4 양자화**: q_max를 127 → 7로 바꾼 `quantize4()`를 만들어 Part 6 분류기의 정확도가 버티는지 실험하세요.
3. **per-channel 맛보기**: Part 6의 `W_fc` 두 열(뉴런 2개)에 각자 scale을 주고 오차를 비교하세요.
4. **MyObserver 업그레이드**: min/max 대신 **히스토그램 기반**(상위 0.1% 무시)으로 r_max를 정하면
   이상치에 강해지는지 실험하세요.

---

수고하셨습니다! 🎉 이제 여러분은 양자화를 "프레임워크가 해주는 마법"이 아니라
**"내가 5분이면 다시 짤 수 있는 산수"**로 이해하게 되었습니다.
다음 실습 「PyTorch 양자화 미니랩 — PTQ 7단계」에서 같은 일이 ResNet 규모에서 벌어지는 것을 확인하세요.
